# 📚 AI Research Paper Intelligence System

A pipeline for semantic search, summarization, keyword extraction, and scientific NER over ML research papers.

**Pipeline:** Load data → Preprocess → Embed → Index (FAISS) → Search → Summarize → Extract keywords → Extract entities → Full combined pipeline

## 1. Setup & Dependencies

Install required libraries.

In [ ]:
!pip install datasets

## 2. Load the Dataset

Loading the `CShorten/ML-ArXiv-Papers` dataset from the Hugging Face Hub.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("CShorten/ML-ArXiv-Papers",split='train')

In [ ]:
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


## 3. Data Preprocessing

Convert to a DataFrame, keep only the relevant columns, and combine `title` + `abstract` into a single text field for embedding.

In [ ]:
import pandas as pd

In [ ]:
df= pd.DataFrame(dataset)
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [ ]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')

In [ ]:
df.shape

(117592, 2)

### 3.1 Subsetting the corpus

Using the first 15,000 papers to keep embedding and indexing fast for this project.

In [ ]:
df=df.head(15000).copy()
df.shape

(15000, 3)

In [ ]:
df.isnull().sum()

,0
title,0
abstract,0


### 3.2 Building the combined text field

Concatenating title and abstract into `paper_text`, then cleaning whitespace/newlines.

In [ ]:
df["paper_text"] = df["title"]+" "+df["abstract"]
df[["paper_text"]].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [ ]:
type(df[["paper_text"]])

pandas.core.frame.DataFrame

In [ ]:
print(df[["paper_text"]].iloc[0])

paper_text    Learning from compressed observations   The pr...
Name: 0, dtype: object


## 4. Sentence Embeddings

Using `all-MiniLM-L6-v2` to convert paper text into dense vector embeddings for semantic similarity search.

In [ ]:
from sentence_transformers import SentenceTransformer

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


### 4.1 Testing embeddings on a single sample

In [ ]:
sample_text=df['paper_text'].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate functions. The ideas are\nillustrated on the example of nonparam

In [ ]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

In [ ]:
sample_text = df["paper_text"].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regres

In [ ]:
embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [ ]:
embedding[:56]

array([-0.1315641 , -0.00678266, -0.00367612,  0.03265158,  0.11219642,
        0.01227267,  0.09816719, -0.0900523 ,  0.04231161, -0.01977348,
       -0.03308417,  0.07452948,  0.10632038, -0.02060429, -0.02052106,
        0.00169493,  0.07081953,  0.05854454, -0.11231912,  0.02082474,
        0.05692544,  0.0201578 ,  0.0258311 ,  0.0321703 ,  0.10513764,
       -0.09676763,  0.02700802, -0.0234509 , -0.04549678, -0.01013699,
       -0.01794855, -0.04814427,  0.01077652, -0.03759069,  0.01943481,
        0.03715189,  0.02967844,  0.04330941,  0.04373213,  0.03704866,
       -0.00182594,  0.00455183, -0.00799067,  0.03037368, -0.014378  ,
        0.03795147,  0.0595916 , -0.02583356, -0.06521576,  0.05900268,
       -0.02107134,  0.07359422, -0.05720106,  0.00294061,  0.00767515,
       -0.0333416 ], dtype=float32)

In [ ]:
sample_embedding=model.encode(df["paper_text"].head(5).to_list())
print(sample_embedding.shape)

(5, 384)


### 4.2 Testing embeddings on a small batch + cosine similarity

Sanity-checking that similarity scores behave as expected before scaling up.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[0].reshape(1,-1))
print(similarity)

[[1.0000001]]


In [ ]:
for i in range(1,5):
  sim=cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[i].reshape(1,-1))
  print(sim)

[[0.36625272]]
[[0.33522844]]
[[0.15505108]]
[[0.37421533]]


### 4.3 Generating embeddings for the full corpus

Embeds all 15,000 papers, with caching to `paper_embeddings.npy` so this only runs once.

In [ ]:
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save("paper_embeddings.npy", embeddings)
    print("Embeddings saved successfully!")

Loading saved embeddings


In [ ]:
print(embedding.shape)
print(type(embedding))

(384,)
<class 'numpy.ndarray'>


In [ ]:
embedding.dtype

dtype('float32')

## 5. Building the FAISS Index

Creating a cosine-similarity search index (`IndexFlatIP` over L2-normalized vectors), cached to `paper_faiss.index`.

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)
    faiss.write_index(index, "paper_faiss.index")
    print("FAISS index saved successfully!")

Loading existing FAISS index


In [ ]:
print(index.ntotal)

15000


### 5.1 Running a test query

Encoding a sample query and retrieving the top-k most similar papers.

In [ ]:
faiss.normalize_L2(query_embedding)
D,I=index.search(query_embedding,5)
print(D)
print(I)
print(df.iloc[11873]["title"])

[[0.6807244  0.67092204 0.65219975 0.62811744 0.61311525]]
[[10466 13730 11873 12691 11282]]
Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection


In [ ]:
print(df.iloc[11873]["abstract"])

  The classification of MRI images according to the anatomical field of view is
a necessary task to solve when faced with the increasing quantity of medical
images. In parallel, advances in deep learning makes it a suitable tool for
computer vision problems. Using a common architecture (such as AlexNet)
provides quite good results, but not sufficient for clinical use. Improving the
model is not an easy task, due to the large number of hyper-parameters
governing both the architecture and the training of the network, and to the
limited understanding of their relevance. Since an exhaustive search is not
tractable, we propose to optimize the network first by random search, and then
by an adaptive search based on Gaussian Processes and Probability of
Improvement. Applying this method on a large and varied MRI dataset, we show a
substantial improvement between the baseline network and the final one (up to
20\% for the most difficult classes).



In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape

(1, 384)

### 5.2 Wrapping search into a reusable function

In [ ]:
def search_paper(query, k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,5)
  for score, idx in zip(D[0],I[0]):
    print("Similarity score", score)
    print("Title", df.iloc[idx]["title"])
    print("Title", df.iloc[idx]["abstract"][:500])
    print()

In [ ]:
search_paper("deep learning for medical image analysis")

Similarity score 0.6807244
Title A Perspective on Deep Imaging
Title   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity score 0.67092204
Title Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Title   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for instance, a large

## 6. Abstractive Summarization

Using `facebook/bart-large-cnn` to generate short summaries of retrieved abstracts.

In [ ]:
!pip install transformers==4.46.3

In [ ]:
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

In [ ]:
summary= summarizer(df.iloc[10466]["abstract"], max_length=120,min_length=40)

In [ ]:
type(summary)

list

In [ ]:
type(summary[0])

dict

In [ ]:
print(summary[0]["summary_text"])

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.


### 6 Combining search + summarization into one function

In [ ]:
def search_paper_and_summarize(query, k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,5)
  for score, idx in zip(D[0],I[0]):
    print("Similarity score", score)
    print("Title", df.iloc[idx]["title"])
    print("Title", df.iloc[idx]["abstract"][:500])
    print()
    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40, do_sample=False)
    print(summary)
    print(summary[0]["summary_text"])
    print()

In [ ]:
search_paper_and_summarize("Deep Learning in Medical Imaging", k=5)

Similarity score 0.73549855
Title A Perspective on Deep Imaging
Title   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]
The combination of tomographic imaging and deep learning, or mach

## 7. Keyword Extraction

Using KeyBERT (built on the same embedding model) to pull out representative keywords/keyphrases from abstracts.

In [ ]:
pip install keybert==0.8.5

In [ ]:
from keybert import KeyBERT

In [ ]:
kw_model = KeyBERT(model)#model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
type(kw_model)

keybert._model.KeyBERT

In [ ]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [ ]:
text=df.iloc[10466]["abstract"]

In [ ]:
keywords = kw_model.extract_keywords(text)
print(keywords)

In [ ]:
print(type(keywords))

<class 'list'>


### 7.1 Extracting multi-word keyphrases

Using n-gram ranges and stopword filtering for more meaningful phrases (e.g., "medical imaging" instead of just "imaging").

In [ ]:
keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1,3),stop_words="english")
print(keywords)

[('tomographic imaging deep', 0.6704), ('imaging deep learning', 0.6543), ('learning medical imaging', 0.6041), ('imaging deep', 0.5919), ('medical imaging', 0.5281)]


In [ ]:
def search_paper_and_summarize(query, k=5):
  query_embedding=model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,5)
  for score, idx in zip(D[0],I[0]):
    print("Similarity score", score)
    print("Title", df.iloc[idx]["title"])
    print("Title", df.iloc[idx]["abstract"][:500])
    print()
    summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40, do_sample=False)
    print(summary)
    print(summary[0]["summary_text"])
    print()
    keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1,3),stop_words="english")
    print(keywords)
    for k, s in keywords:
      print(k)


## 8. Scientific Named Entity Recognition (NER)

Using `RJuro/SciNERTopic` to tag domain-specific entities (e.g., `Method`, `Task`) in paper abstracts.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

sci_ner_tokenizer = AutoTokenizer.from_pretrained("RJuro/SciNERTopic")
sci_ner_model = AutoModelForTokenClassification.from_pretrained("RJuro/SciNERTopic")

ner_pipeline = pipeline(
    "ner",
    model=sci_ner_model,
    tokenizer=sci_ner_tokenizer,
    aggregation_strategy="average"  # recommended by the model card; merges sub-word tokens
)

tokenizer_config.json:   0%|          | 0.00/421 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

In [ ]:
type(ner_pipeline)

transformers.pipelines.token_classification.TokenClassificationPipeline

In [ ]:
text = df.iloc[10466]["abstract"]
print(text)

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [ ]:
entities = ner_pipeline(text)
print(entities)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity_group': 'Task', 'score': np.float32(0.7357169), 'word': 'tomographic imaging', 'start': 21, 'end': 40}, {'entity_group': 'Method', 'score': np.float32(0.997051), 'word': 'deep learning', 'start': 45, 'end': 58}, {'entity_group': 'Method', 'score': np.float32(0.9972993), 'word': 'machine learning', 'start': 63, 'end': 79}, {'entity_group': 'Task', 'score': np.float32(0.9946006), 'word': 'image analysis', 'start': 121, 'end': 135}, {'entity_group': 'Task', 'score': np.float32(0.99623466), 'word': 'image reconstruction', 'start': 145, 'end': 165}, {'entity_group': 'Task', 'score': np.float32(0.9878021), 'word': 'medical imaging', 'start': 247, 'end': 262}, {'entity_group': 'Method', 'score': np.float32(0.9655553), 'word': 'image reconstruction theories and techniques', 'start': 294, 'end': 338}, {'entity_group': 'OtherScientificTerm', 'score': np.float32(0.9922292), 'word': 'domain knowledge', 'start': 396, 'end': 412}, {'entity_group': 'Material', 'score': np.float32(0.7865238)

In [ ]:
for ent in entities:
    print(ent["word"], "->", ent["entity_group"], f"(score: {ent['score']:.2f})")

tomographic imaging -> Task (score: 0.74)
deep learning -> Method (score: 1.00)
machine learning -> Method (score: 1.00)
image analysis -> Task (score: 0.99)
image reconstruction -> Task (score: 1.00)
medical imaging -> Task (score: 0.99)
image reconstruction theories and techniques -> Method (score: 0.97)
domain knowledge -> OtherScientificTerm (score: 0.99)
big data -> Material (score: 0.79)
approaches -> Generic (score: 0.96)
image reconstruction -> Task (score: 1.00)
clinical and preclinical applications -> Task (score: 0.97)
machine learning -> Method (score: 1.00)
medical imaging -> Task (score: 1.00)


In [ ]:
def extract_entities(text, min_score=0.4):
    raw_entities = ner_pipeline(text)
    seen = set()
    cleaned = []
    for ent in raw_entities:
        if ent["score"] < min_score:
            continue
        key = (ent["word"].lower(), ent["entity_group"])
        if key in seen:
            continue
        seen.add(key)
        cleaned.append({
            "text": ent["word"],
            "label": ent["entity_group"],
            "score": round(float(ent["score"]), 3)
        })
    return cleaned

In [ ]:
extract_entities(df.iloc[10466]["abstract"])

[{'text': 'tomographic imaging', 'label': 'Task', 'score': 0.736},
 {'text': 'deep learning', 'label': 'Method', 'score': 0.997},
 {'text': 'machine learning', 'label': 'Method', 'score': 0.997},
 {'text': 'image analysis', 'label': 'Task', 'score': 0.995},
 {'text': 'image reconstruction', 'label': 'Task', 'score': 0.996},
 {'text': 'medical imaging', 'label': 'Task', 'score': 0.988},
 {'text': 'image reconstruction theories and techniques',
  'label': 'Method',
  'score': 0.966},
 {'text': 'domain knowledge', 'label': 'OtherScientificTerm', 'score': 0.992},
 {'text': 'big data', 'label': 'Material', 'score': 0.787},
 {'text': 'approaches', 'label': 'Generic', 'score': 0.96},
 {'text': 'clinical and preclinical applications',
  'label': 'Task',
  'score': 0.965}]

## 9. Full Pipeline: Search → Summarize → Tag

The complete end-to-end function: given a query, retrieve top-k papers, summarize each abstract, and extract scientific entities — returning structured results.

In [ ]:
def search_summarize_and_tag(query, k=5):
    """Full pipeline: semantic search -> summarize -> extract scientific entities."""
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        print("Similarity score:", score)
        print("Title:", df.iloc[idx]["title"])
        abstract = df.iloc[idx]["abstract"]
        print("Abstract:", abstract[:500])
        print()

        summary = summarizer(abstract, max_length=120, min_length=40, do_sample=False)
        print("Summary:", summary[0]["summary_text"])
        print()

        entities = extract_entities(abstract)
        print("Scientific entities:")
        if entities:
            for ent in entities:
                print(f"  - {ent['text']}  ->  {ent['label']}  (score: {ent['score']})")
        else:
            print("  (no entities above confidence threshold)")
        print("-" * 80)

In [ ]:
search_summarize_and_tag("Deep Learning in Medical Imaging", k=5)

Similarity score: 0.73549855
Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Summary: The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Scientific entities:
  - tomographic imaging  ->  Task  (score: 0.736)
